# Mistral Document AI - OCR Extraction Demo

This notebook demonstrates **Mistral Document AI** on **Microsoft Foundry** — extracting text,
tables, and images from PDF documents via the REST API.

## Available Models

| Model | Version | Key Features |
|-------|---------|-------------|
| `mistral-document-ai-2512` | v25.12 | OCR, images, bbox annotations, table_format, header/footer |
| `mistral-ocr-4-0` | 4.0 (Preview) | + paragraph bounding boxes, block classification, confidence scores, 170 languages |

## Prerequisites

1. Deploy both models: `./scripts/deploy_all.ps1` or `./scripts/deploy_all.sh`
2. Generate `.env` from existing deployments: `./scripts/setup_env.ps1` or `./scripts/setup_env.sh`
   (or copy `.env.example` to `.env` and fill in manually)
3. Place PDF files in the `data/` folder
4. Install: `uv sync --group notebook`
5. Run: `uv run jupyter lab demo_ocr.ipynb`

## What this notebook covers

| Step | Description |
|------|-------------|
| —    | Model comparison table + **API limits** (30 MB / 30 pages on Azure) |
| 1    | Configuration and authentication setup |
| 2    | Model selection (v25.12 or OCR 4.0) |
| 3    | PDF file selection from `data/` |
| 4    | OCR extraction via Mistral Document AI REST API |
| 4b   | **v25.12 / OCR 4.0**: table_format, extract_header, extract_footer |
| 4c   | **OCR 4.0 only**: content blocks (bbox + type) + confidence scores |
| 5    | Markdown rendering and preview |
| 6    | Table detection and DataFrame conversion |
| 7    | Image inspection and display |
| 8    | Per-page analysis breakdown |
| 9    | Annotations demo (bbox + document-level) |
| 10   | Model comparison (v25.12 vs OCR 4.0) |
| 11   | Save results to `extraction/` |
| 12   | Agent-style workflow pattern |

## v25.12 vs OCR 4.0 — Key Differences

Mistral Document AI is available in two versions on **Microsoft Foundry**. Both are deployed as **GlobalStandard** SKU endpoints in the same Foundry project (OCR 4.0 is Preview).

| Feature | v25.12 (`mistral-document-ai-2512`) | OCR 4.0 (`mistral-ocr-4-0`) |
|---------|--------------------------------------|--------------------------------------|
| **OCR engine** | `mistral-ocr-2512` + `mistral-small-2506` | `mistral-ocr-4-0` (+ Mistral Medium 3.5) |
| **Status** | GA | Preview |
| **`table_format`** | `"markdown"` / `"html"` | `"markdown"` / `"markdown-tables"` / `"html"` (colspan/rowspan) |
| **`extract_header` / `extract_footer`** | Supported | Supported |
| **`pages` selection** | Select specific pages (0-indexed) | Supported |
| **`image_limit`** | Cap number of returned images | Supported |
| **Paragraph bounding boxes** | Not supported | Per-paragraph layout coordinates |
| **Block classification** | Not supported | title, header, footer, code, table, equation, paragraph, list, signature, image, caption, references |
| **Inline confidence scores** | Not supported | Per-page / per-word confidence |
| **Streaming API** | Not supported | Redesigned, reduced time-to-first-token |
| **Multilingual** | 99%+ across 25+ languages | **170 languages** |

> **Sources**: [Microsoft Foundry Blog — Mistral Document AI with OCR 4 and Mistral Medium 3.5](https://techcommunity.microsoft.com/blog/azure-ai-foundry-blog/mistral-document-ai-with-ocr-4-and-mistral-medium-3-5-arrive-in-microsoft-foundr/4529863) · [Mistral OCR 4](https://mistral.ai/news/ocr-4/)

**In short**: OCR 4.0 adds paragraph-level bounding boxes, block classification, inline confidence scores, 170-language support, and a streaming-capable API on top of v25.12's structured extraction. Use v25.12 for stable GA workloads, and OCR 4.0 for the richest layout understanding.

---

### API Limits

| Platform | Max File Size | Max Pages / Request | Annotations Limit |
|----------|--------------|---------------------|--------------------|
| **Microsoft Foundry** | 30 MB | 30 pages | 8 pages |
| **Mistral La Plateforme** | 50 MB | 1 000 pages | 8 pages |

- **Auto-chunking**: For PDFs exceeding 30 pages on Azure, `ocr_pdf()` in `src/extract.py` automatically splits the document into ≤30-page chunks and merges results.
- **Annotations** (`bbox_annotation_format`, `document_annotation_format`): Only the first **8 pages** are annotated per request on both platforms.
- **Rate limits**: Microsoft Foundry applies per-deployment token/request rate limits depending on the SKU provisioning (GlobalStandard = pay-per-token, no reserved capacity).

> **Tip**: For very large documents, use the v25.12 `pages` parameter to extract only the pages you need.

In [ ]:
# %% Cell 1: Setup and Configuration
# Load environment variables and verify connectivity
import os
import json
import base64
import re
from pathlib import Path
from IPython.display import display, Markdown, Image

# Load .env file (same pattern used in app.py and extract.py)
env_path = Path(".env")
if env_path.exists():
    for line in env_path.read_text().splitlines():
        line = line.strip()
        if line and not line.startswith("#") and "=" in line:
            k, _, v = line.partition("=")
            os.environ.setdefault(k.strip(), v.strip())
    print(".env loaded")
else:
    print("No .env found - copy .env.example to .env and fill in values")

# Read configuration
endpoint = os.environ.get("MISTRAL_ENDPOINT", "")
api_version = os.environ.get("MISTRAL_API_VERSION", "2024-05-01-preview")
api_key = os.environ.get("AZURE_AI_KEY", "")

# Display config (mask sensitive values)
print(f"\nEndpoint   : {endpoint[:50]}..." if len(endpoint) > 50 else f"\nEndpoint   : {endpoint}")
print(f"API Version: {api_version}")
print(f"Auth       : {'API Key (set)' if api_key else 'Azure AD (DefaultAzureCredential)'}")

assert endpoint, "MISTRAL_ENDPOINT is required! Set it in .env or environment."
print("\nConfiguration OK")

## Step 2 - Select Model Version

Choose between `v25.12` (GA; adds `table_format`, header/footer extraction) and `OCR 4.0` (Preview; adds bounding boxes, block classification, confidence scores).
Both share the same endpoint and API key.

In [ ]:
# %% Cell 2: Model selection
from src.extract import MODELS, get_deployment_name
import pandas as pd

# Display available models
models_df = pd.DataFrame([
    {"Key": k, "Label": v["label"], "Deployment": v["default_deployment"],
     "Description": v["description"]}
    for k, v in MODELS.items()
])
display(models_df)

# Choose model: "ocr4" (OCR 4.0, Preview) or "2512" (table_format, headers, footers)
MODEL_KEY = "ocr4"

deployment = get_deployment_name(MODEL_KEY)
print(f"\nSelected: {MODELS[MODEL_KEY]['label']} -> deployment '{deployment}'")

## Step 3 - Select a PDF

List PDFs in the `data/` folder and pick one for processing.
The sample PDF `sample_complex_report.pdf` contains 11 tables across 5 pages
to demonstrate table extraction capabilities.

In [ ]:
# %% Cell 2: List and select PDF files
data_dir = Path("data")
pdfs = sorted(data_dir.glob("*.pdf"))

if not pdfs:
    print("No PDFs found in data/ - place some PDF files there first!")
    print("Tip: run `uv run python scripts/generate_sample_pdf.py` to create a sample")
else:
    print(f"Found {len(pdfs)} PDF(s) in data/:\n")
    for i, p in enumerate(pdfs):
        size_kb = p.stat().st_size / 1024
        print(f"  [{i}] {p.name:40s} ({size_kb:7.1f} KB)")

# Select the first PDF (change PDF_INDEX to pick a different one)
PDF_INDEX = 0
pdf_path = pdfs[PDF_INDEX] if pdfs else None
print(f"\nSelected: {pdf_path}")

## Step 4 - Call Mistral Document AI (OCR)

The OCR endpoint:
- URL: `POST {endpoint}/providers/mistral/azure/ocr?api-version=2024-05-01-preview`
- Body: `{"model": "...", "document": {"type": "document_url", "document_url": "data:application/pdf;base64,..."}}`
- Auth: `Bearer` token (API key or Azure AD)

The `model_key` selects which deployment to use. For large PDFs (>30 pages), `ocr_pdf()` auto-splits.

In [ ]:
# %% Cell 4: Run OCR extraction
from src.extract import ocr_pdf

assert pdf_path, "No PDF selected - run the cell above first"

result = await ocr_pdf(pdf_path, model_key=MODEL_KEY, include_images=True)

print(f"Model           : {result.model}")
print(f"Pages extracted : {len(result.pages)}")
print(f"Total characters: {len(result.markdown):,}")
print(f"Total words     : {len(result.markdown.split()):,}")
print(f"Images detected : {len(result.images)}")
print(f"Token usage     : {result.usage}")
print(f"Elapsed         : {result.elapsed_ms:.0f} ms")

## Step 4b — Advanced (v25.12 / OCR 4.0): Structured Table Extraction + Headers/Footers

Both `mistral-document-ai-2512` and `mistral-ocr-4-0` support three key parameters:
- **`table_format`**: `"markdown"` or `"html"` — returns tables as structured output rather than inline markdown
- **`extract_header`**: pulls page-level headers
- **`extract_footer`**: pulls page-level footers

This cell runs an OCR call with these features enabled and compares the output.

In [ ]:
# %% Cell 4b: v25.12 — table_format + headers/footers
from src.extract import ocr_pdf, MODELS

assert pdf_path, "No PDF selected - run step 3 first"

# Run v25.12 with all exclusive features enabled
result_2512 = await ocr_pdf(
    pdf_path,
    model_key="2512",
    include_images=True,
    table_format="markdown",  # or "html" — v25.12 only
    extract_header=True,       # v25.12 only
    extract_footer=True,       # v25.12 only
)

print(f"Model           : {result_2512.model}")
print(f"Pages extracted : {len(result_2512.pages)}")
print(f"Characters      : {len(result_2512.markdown):,}")
print(f"Images          : {len(result_2512.images)}")
print(f"Elapsed         : {result_2512.elapsed_ms:.0f} ms")

# Show headers and footers per page (v25.12 exclusive)
print(f"\n{'='*60}")
print("Headers & Footers (v25.12 only)")
print(f"{'='*60}")
for p in result_2512.pages:
    header = p.header or "(none)"
    footer = p.footer or "(none)"
    print(f"  Page {p.page_index + 1}: header={header!r}, footer={footer!r}")

# Show structured tables from the API response (v25.12 table_format)
api_tables = []
for p in result_2512.pages:
    for tbl in p.tables:
        api_tables.append({"page": p.page_index + 1, "table": tbl})

if api_tables:
    print(f"\nStructured tables from API (table_format='markdown'): {len(api_tables)}")
    for i, t in enumerate(api_tables[:3]):  # preview first 3
        print(f"\n--- Table {i + 1} (page {t['page']}) ---")
        content = t["table"].get("markdown") or t["table"].get("content", "")
        print(content[:500] if isinstance(content, str) else json.dumps(content, indent=2)[:500])
else:
    print("\nNo structured tables returned via API — tables are in the markdown output.")

## Step 4c — OCR 4.0 exclusive: Content Blocks + Confidence Scores

`mistral-ocr-4-0` adds two parameters not available in v25.12:
- **`include_blocks=True`** — returns `pages[].blocks`: paragraph-level content blocks, each with a **bounding box** and a **type** (title, paragraph, table, equation, signature, ...). Ideal for layout-aware pipelines and semantic chunking (RAG).
- **`confidence_scores_granularity="page"`** (or `"word"`) — returns `pages[].confidence_scores` for human-in-the-loop QA and automated error flagging.

In [ ]:
# %% Cell 4c: OCR 4.0 — content blocks (bbox + type) + confidence scores
from src.extract import ocr_pdf

assert pdf_path, "No PDF selected - run step 3 first"

# OCR 4.0 exclusive parameters -> populate pages[].blocks and pages[].confidence_scores
result_ocr4 = await ocr_pdf(
    pdf_path,
    model_key="ocr4",
    include_images=False,
    include_blocks=True,                    # OCR 4.0 only — bbox + block type
    confidence_scores_granularity="page",   # OCR 4.0 only — "page" or "word"
)

print(f"Model           : {result_ocr4.model}")
print(f"Pages extracted : {len(result_ocr4.pages)}")

# Content blocks: paragraph-level type classification + bounding boxes
total_blocks = sum(len(p.blocks) for p in result_ocr4.pages)
print(f"Content blocks  : {total_blocks}")
for p in result_ocr4.pages:
    if not p.blocks:
        continue
    types = {}
    for b in p.blocks:
        types[b.get('type', '?')] = types.get(b.get('type', '?'), 0) + 1
    print(f"  Page {p.page_index + 1} block types: {types}")
    break

# Inline confidence scores (per-page or per-word)
for p in result_ocr4.pages:
    if p.confidence_scores is not None:
        print(f"\nConfidence scores (page {p.page_index + 1}): {p.confidence_scores}")
        break

if total_blocks == 0:
    print("\nNote: blocks/confidence_scores are returned only by OCR 4.0, and only when requested via the parameters above.")

## Step 5 - View Extracted Markdown

Render the OCR output as Markdown directly in the notebook.
Toggle between a preview (first 3000 chars) and full output.

In [ ]:
# %% Cell 4: Render extracted markdown
# Set PREVIEW_ONLY = False to render FULL output
PREVIEW_ONLY = True
MAX_PREVIEW = 3000

if PREVIEW_ONLY:
    preview = result.markdown[:MAX_PREVIEW]
    if len(result.markdown) > MAX_PREVIEW:
        preview += f"\n\n---\n*... truncated ({len(result.markdown) - MAX_PREVIEW:,} chars remaining) ...*"
    display(Markdown(preview))
else:
    display(Markdown(result.markdown))

## Step 6 - Extract Tables from Markdown

Mistral Document AI outputs tables in Markdown pipe format (`|col1|col2|`).
This step parses them into pandas DataFrames for analysis and export.

In [ ]:
# %% Cell 5: Parse and display tables
import pandas as pd


def extract_markdown_tables(md_text: str) -> list[pd.DataFrame]:
    """Parse Markdown pipe tables into DataFrames."""
    tables = []
    lines = md_text.split("\n")
    i = 0
    while i < len(lines):
        line = lines[i].strip()
        if "|" in line and i + 1 < len(lines) and re.match(r"^[\|\s:\-]+$", lines[i + 1].strip()):
            header = [c.strip() for c in line.strip("|").split("|")]
            rows = []
            j = i + 2
            while j < len(lines) and "|" in lines[j] and lines[j].strip():
                row = [c.strip() for c in lines[j].strip("|").split("|")]
                rows.append(row)
                j += 1
            if rows:
                df = pd.DataFrame(rows, columns=header[:len(rows[0])])
                tables.append(df)
            i = j
        else:
            i += 1
    return tables


tables = extract_markdown_tables(result.markdown)
print(f"Tables found: {len(tables)}\n")

# Summary of all tables
summary = pd.DataFrame([
    {"Table": i + 1, "Rows": df.shape[0], "Cols": df.shape[1],
     "Columns": ", ".join(df.columns[:4]) + ("..." if df.shape[1] > 4 else "")}
    for i, df in enumerate(tables)
])
display(summary)

In [ ]:
# %% Cell 5b: Display individual tables
# Show each table with its full content
for idx, df in enumerate(tables):
    print(f"\n{'='*60}")
    print(f"Table {idx + 1} ({df.shape[0]} rows x {df.shape[1]} cols)")
    print(f"{'='*60}")
    display(df)

## Step 7 - Inspect Extracted Images

If `include_images=True`, Mistral returns base64-encoded images with metadata.
Each image includes its page index and an identifier.

In [ ]:
# %% Cell 6: Display extracted images
if result.images:
    print(f"Total images: {len(result.images)}\n")
    for img in result.images[:5]:  # Preview up to 5
        img_id = img.get("id", "unknown")
        page = img.get("page_index", "?")
        b64 = img.get("image_base64") or img.get("base64", "")
        print(f"Image: {img_id} (page {page})")
        print(f"  Keys: {list(img.keys())}")
        if b64:
            img_bytes = base64.b64decode(b64)
            display(Image(data=img_bytes, width=400))
        else:
            print("  (no base64 data)")
    if len(result.images) > 5:
        print(f"\n... and {len(result.images) - 5} more images")
else:
    print("No images extracted.")
    print("This is expected for text-only PDFs (tables/text only, no embedded images).")

## Step 8 - Per-Page Analysis

Break down the extraction results by page to understand the document structure.

In [ ]:
# %% Cell 8: Per-page breakdown
page_data = []
for p in result.pages:
    page_tables = extract_markdown_tables(p.markdown)
    row = {
        "Page": p.page_index + 1,
        "Characters": len(p.markdown),
        "Words": len(p.markdown.split()),
        "Tables": len(page_tables) + len(p.tables),
        "Images": len(p.images),
        "Preview": p.markdown[:80].replace("\n", " "),
    }
    if p.header:
        row["Header"] = p.header[:40]
    if p.footer:
        row["Footer"] = p.footer[:40]
    page_data.append(row)

page_df = pd.DataFrame(page_data)
display(page_df)

# Show the page with the most tables
if not page_df.empty:
    densest = page_df.loc[page_df["Tables"].idxmax()]
    print(f"\nDensest page: Page {int(densest['Page'])} with {int(densest['Tables'])} table(s)")

## Step 9 - Annotations Demo

Mistral supports **bbox annotations** (per-image structured data) and **document annotations** (whole-document summary).

- `bbox_annotation_format`: JSON Schema applied to each image (uses `ImageDescription`)
- `document_annotation_format`: JSON Schema for the entire document
- `document_annotation_prompt`: Optional prompt guiding the document annotation

In [ ]:
# %% Cell 9: Annotations - bbox and document-level
from src.extract import _pydantic_to_mistral_schema, ImageDescription
from pydantic import BaseModel, Field

# Define a document-level annotation schema
class DocumentSummary(BaseModel):
    topics: list[str] = Field(..., description="Key topics covered in the document")
    entities: list[str] = Field(..., description="Named entities (companies, people, products)")
    summary: str = Field(..., description="One-paragraph executive summary")

# Build annotation schemas
bbox_schema = _pydantic_to_mistral_schema(ImageDescription)
doc_schema = _pydantic_to_mistral_schema(DocumentSummary)

print("BBox annotation schema (per-image):")
print(json.dumps(bbox_schema, indent=2)[:300], "...")
print("\nDocument annotation schema (whole-doc):")
print(json.dumps(doc_schema, indent=2)[:300], "...")

# Run OCR with annotations
assert pdf_path, "No PDF selected"
annotated_result = await ocr_pdf(
    pdf_path,
    model_key=MODEL_KEY,
    include_images=True,
    bbox_annotation_format=bbox_schema,
    document_annotation_format=doc_schema,
    document_annotation_prompt="Summarize the document with topics, entities, and a brief summary.",
)

print(f"\nDocument annotation: {annotated_result.document_annotation}")
print(f"Images with annotations: {len(annotated_result.images)}")

## Step 10 - Model Comparison (v25.12 vs OCR 4.0)

Run the same PDF through both models and compare results: pages, characters, words,
tables detected, images, elapsed time, and token usage.

In [ ]:
# %% Cell 10: Model comparison - v25.12 vs OCR 4.0
assert pdf_path, "No PDF selected"

comparison_rows = []
for mk in MODELS:
    print(f"Running {MODELS[mk]['label']}...")
    r = await ocr_pdf(pdf_path, model_key=mk, include_images=True)
    md_tables = extract_markdown_tables(r.markdown)
    comparison_rows.append({
        "Model": MODELS[mk]["label"],
        "Deployment": r.model,
        "Pages": len(r.pages),
        "Characters": len(r.markdown),
        "Words": len(r.markdown.split()),
        "Tables (md)": len(md_tables),
        "Tables (api)": sum(len(p.tables) for p in r.pages),
        "Images": len(r.images),
        "Elapsed (ms)": round(r.elapsed_ms),
        "Tokens": r.usage.get("total_tokens", r.usage.get("pages_processed", "?")),
    })

comp_df = pd.DataFrame(comparison_rows)
display(comp_df)

# Highlight differences
if len(comparison_rows) == 2:
    a, b = comparison_rows
    for key in ["Characters", "Words", "Tables (md)", "Images"]:
        if a[key] != b[key]:
            print(f"  {key}: {a['Model']}={a[key]}, {b['Model']}={b[key]}")

## Step 11 - Save Results

Persist the extraction to the `extraction/` folder:
- `{stem}.md` - Full markdown output
- `{stem}_table_{n}.csv` - Each table as CSV
- `{stem}_meta.json` - Metadata (pages, tokens, timing)
- `{stem}_full.json` - Complete OCR result

In [ ]:
# %% Cell 11: Save results to extraction/
out_dir = Path("extraction")
out_dir.mkdir(exist_ok=True)
stem = pdf_path.stem

# Save markdown
md_out = out_dir / f"{stem}.md"
md_out.write_text(result.markdown, encoding="utf-8")
print(f"Markdown: {md_out}")

# Save tables as CSV
for i, df in enumerate(tables):
    csv_out = out_dir / f"{stem}_table_{i + 1}.csv"
    df.to_csv(csv_out, index=False)
    print(f"Table {i + 1} : {csv_out}")

# Save metadata
meta = {
    "source": pdf_path.name,
    "model": result.model,
    "pages": len(result.pages),
    "images_count": len(result.images),
    "chars": len(result.markdown),
    "words": len(result.markdown.split()),
    "tables_found": len(tables),
    "usage": result.usage,
    "elapsed_ms": round(result.elapsed_ms, 1),
}
meta_out = out_dir / f"{stem}_meta.json"
meta_out.write_text(json.dumps(meta, indent=2, ensure_ascii=False), encoding="utf-8")
print(f"Metadata: {meta_out}")

# Save full JSON result
full_out = out_dir / f"{stem}_full.json"
full_json = result.model_dump()
full_json["source"] = pdf_path.name
full_out.write_text(json.dumps(full_json, indent=2, ensure_ascii=False), encoding="utf-8")
print(f"Full JSON: {full_out}")

print(f"\nAll results saved in {out_dir}/")

## Step 12 - Agent-Style OCR Workflow

A minimal "agent" pattern demonstrating a production-ready flow:
1. Load PDF
2. Extract with OCR (with model selection and annotations)
3. Structure the output (tables, images, metadata)
4. Return structured result

In [ ]:
# %% Cell 12: Agent workflow pattern
async def ocr_agent_workflow(pdf_file: Path, model_key: str = "ocr4") -> dict:
    """Agent workflow: PDF -> OCR -> structured output."""
    print(f"[1/3] Loading {pdf_file.name}...")

    print(f"[2/3] Running Mistral Document AI OCR ({MODELS[model_key]['label']})...")
    result = await ocr_pdf(pdf_file, model_key=model_key, include_images=True)

    print("[3/3] Structuring output...")
    extracted_tables = extract_markdown_tables(result.markdown)

    output = {
        "source": pdf_file.name,
        "model": result.model,
        "markdown": result.markdown,
        "pages": len(result.pages),
        "tables": [df.to_dict(orient="records") for df in extracted_tables],
        "images": [
            {
                "id": img.get("id"),
                "page": img.get("page_index"),
                "has_base64": bool(img.get("image_base64") or img.get("base64")),
            }
            for img in result.images
        ],
        "usage": result.usage,
        "elapsed_ms": result.elapsed_ms,
    }

    print(f"\nDone: {len(result.pages)} pages, {len(extracted_tables)} tables, "
          f"{len(result.images)} images in {result.elapsed_ms:.0f}ms")
    return output


# Run the workflow
if pdf_path:
    workflow_result = await ocr_agent_workflow(pdf_path, model_key=MODEL_KEY)

    # Preview extracted tables
    for i, tbl in enumerate(workflow_result["tables"]):
        print(f"\nTable {i + 1} ({len(tbl)} rows):")
        display(pd.DataFrame(tbl))